# Convert outputs to yearly zarr files

In [1]:
import re
import os
import sys

import zarr
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
sys.path.insert(0, os.path.realpath('../libs/'))
import verif_utils as vu

### Get the target data for coord reference

In [4]:
# fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2020.zarr'
# ds_target = xr.open_zarr(fn_target)

## Gather prognostic outputs

### Combine raw netCDF4 to zarr

In [5]:
# ind_start = 0
# ind_end = 100


# source_dir = f'/glade/derecho/scratch/ksha/DWC/RAW_OUTPUT/CONUS_GP_CESM_HIST/'
# fn_all = vu.get_nc_files(source_dir)[0]
# fn_all = sorted(fn_all, key=lambda x: int(re.search(r'_(\d+)\.nc$', x).group(1)))

# ds_collect = []

# for fn in fn_all[ind_start:ind_end]:
#     ds = xr.open_dataset(fn)
#     ds_collect.append(ds)

# ds_final = xr.concat(ds_collect, dim='time')

# ds_final = ds_final.rename({'latitude': 'south_north', 'longitude': 'west_east', 'level': 'bottom_top'})
# ds_final['west_east'] = np.arange(336).astype(np.float32)
# ds_final['south_north'] = np.arange(336).astype(np.float32)
# ds_final['bottom_top'] = np.arange(12).astype(np.float32)

### Concat zarr to yearly

In [6]:
def extract_start_index(path):
    # Find numbers like _0000_1000_ and take the first one
    match = re.search(r'_(\d+)_\d+_', path)
    return int(match.group(1)) if match else float('inf')

def year_from_dt64(dt64):
    """Convert numpy.datetime64[ns] to integer year."""
    return dt64.astype("datetime64[Y]").astype(int) + 1970

def first_day_of_year(dt64):
    """Return the datetime64[ns] of the first day of the year for dt64."""
    return dt64.astype("datetime64[Y]").astype("datetime64[ns]")

### CESM

In [7]:
for i_year, year in enumerate(range(2005, 2010)):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_HIST/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (i_year != 0):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(10, 12, 336, 336))
    chunk_size_4d = dict(chunks=(10, 12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_CESM_HIST_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_2005.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_HIST/CESM_HIST_9000_9500_2070-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_2006.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_HIST/CESM_HIST_18000_18500_2070-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_2007.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_HIST/CESM_HIST_26500_27000_2070-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_2008.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_HIST/CESM_HIST_35500_36000_2070-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_i

In [5]:
xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_2009.zarr')

<xarray.Dataset>
Dimensions:        (time: 8729, south_north: 336, west_east: 336, bottom_top: 12)
Coordinates:
  * bottom_top     (bottom_top) float32 0.0 1.0 2.0 3.0 ... 8.0 9.0 10.0 11.0
  * south_north    (south_north) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
  * time           (time) datetime64[ns] 2009-01-01 ... 2009-12-30T16:00:00
  * west_east      (west_east) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
Data variables: (12/13)
    WRF_MSLP       (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_P          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_PWAT_05    (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_Q_tot_05   (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_SP         (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_T          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    ...             ...
    WRF_TD2        (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_U          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_U10        (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_V          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_V10        (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    forecast_hour  (time) int64 dask.array<chunksize=(12,), meta=np.ndarray>

In [10]:
year = 2005; i_year = 0

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_HIST/'
fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)

ds_collect = []
flag_add_previous = True

for i_fn, fn in enumerate(fn_all):
    ds = xr.open_zarr(fn)

    # Extract last timestamp
    first_day = ds['time'].values[-1]
    year_ = year_from_dt64(first_day)

    if year_ != year:
        continue  # Skip years not matching target
    
    # If file starts mid-year, add the previous file once
    if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous:
        print(fn)
        ds_prev = xr.open_zarr(fn_all[i_fn - 1])
        ds_collect.append(ds_prev)
        flag_add_previous = False

    ds_collect.append(ds)

# Combine and trim to exact calendar year
if not ds_collect:
    raise ValueError(f"No data found for year {year}")

ds_final = xr.concat(ds_collect, dim='time')

ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-01-02T23:00:00"))
ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336})

# =================================================== #
# zarr encodings
dict_encoding = {}
varnames = list(ds_final.keys())
varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']

chunk_size_3d = dict(chunks=(10, 12, 336, 336))
chunk_size_4d = dict(chunks=(10, 12, 12, 336, 336))
compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

for i_var, var in enumerate(varnames):
    if var in varname_4D:
        dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
    else:
        dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/clim_ref/'
save_name = base_dir + f'init_HIST_{year}.zarr'
ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_HIST/CESM_HIST_43500_44000_2070-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/clim_ref/init_HIST_2005.zarr


In [11]:
ds

<xarray.Dataset>
Dimensions:        (time: 89, south_north: 336, west_east: 336, bottom_top: 12)
Coordinates:
  * bottom_top     (bottom_top) float32 0.0 1.0 2.0 3.0 ... 8.0 9.0 10.0 11.0
  * south_north    (south_north) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
  * time           (time) datetime64[ns] 2005-01-28T05:00:00 ... 2005-01-31T2...
  * west_east      (west_east) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
Data variables: (12/13)
    WRF_MSLP       (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_P          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_PWAT_05    (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_Q_tot_05   (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_SP         (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_T          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    ...             ...
    WRF_TD2        (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_U          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_U10        (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_V          (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_V10        (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    forecast_hour  (time) int64 dask.array<chunksize=(12,), meta=np.ndarray>

In [8]:
for i_year, year in enumerate(range(1980, 1985)):
    
    fn = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_{year}.zarr'
    ds = xr.open_zarr(fn)
    ds = ds[['WRF_T2',]]
    
    out = xr.Dataset()
    out["WRF_T2_min"] = ds['WRF_T2'].resample(time="1D").min(keep_attrs=True)
    out["WRF_T2_max"] = ds['WRF_T2'].resample(time="1D").max(keep_attrs=True)
    
    output_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_{year}_daily.zarr'
    out.to_zarr(output_name, mode='w', consolidated=True, compute=True)
    print(output_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_1980_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_1981_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_1982_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_1983_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_1984_daily.zarr


## CESM-LENS2 daily

In [4]:
for i_year, year in enumerate(range(1980, 1985)):
    
    fn = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/dscale_CESM_HIST/CESM_GP_{year}.zarr'
    ds = xr.open_zarr(fn)
    ds = ds[['VAR_2T',]]
    
    out = xr.Dataset()
    out["VAR_2T_min"] = ds['VAR_2T'].resample(time="1D").min(keep_attrs=True)
    out["VAR_2T_max"] = ds['VAR_2T'].resample(time="1D").max(keep_attrs=True)
    
    output_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_LENS2_HIST_{year}_daily.zarr'
    out.to_zarr(output_name, mode='w', consolidated=True, compute=True)
    print(output_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_LENS2_HIST_1980_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_LENS2_HIST_1981_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_LENS2_HIST_1982_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_LENS2_HIST_1983_daily.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/CESM_LENS2_HIST_1984_daily.zarr
